# RAG-Based AI Assistant — UChicago MS in Applied Data Science
**GEN AI Principles | Course Project 1**


| # | Cell | What it does |
|---|------|-------------|
| 1 | Fix numpy + install | Pins numpy first to avoid binary conflicts, then installs all packages |
| 2 | Credentials | Azure OpenAI keys via Colab Secrets |
| 3 | Setup | Creates folders, adds src/ to path |
| 4 | scraper.py | Write web crawler to disk |
| 5 | embedder.py | Write embedding + vector store module |
| 6 | llm_factory.py | Write LLM factory module |
| 7 | rag_chain.py | Write RAG chain module |
| 8 | evaluator.py | Write evaluation module |
| 9 | Scrape | Crawl 13 MS-ADS pages live (~25 sec) |
| 10 | Inspect | Summary table + sample chunks |
| 11 | Embed | Build ChromaDB vector store |
| 12 | Test retrieval | Verify similarity search |
| 13 | Build RAG chain | Connect retriever + LLM |
| 14 | Q&A | 5 example questions |
| 15 | Multi-turn | Follow-up questions with memory |
| 16 | Evaluation | Heuristic scoring on 10 Q&A pairs |
| 17 | RAGAS | LLM-as-judge evaluation (optional) |
| 18 | Streamlit UI | Launch chatbot via ngrok |


## Cell 1 — Install Dependencies

In [ ]:
# Step 1: pin numpy to avoid binary incompatibility
!pip install -q "numpy>=1.24,<2.0"

# Step 2: install all dependencies — versions confirmed working in Colab
!pip install -q \
    langchain==0.2.16 \
    langchain-community==0.2.16 \
    langchain-core==0.2.38 \
    langchain-openai==0.1.23 \
    langchain-huggingface==0.0.3 \
    chromadb==0.5.5 \
    faiss-cpu==1.8.0 \
    sentence-transformers==3.0.1 \
    requests==2.32.3 \
    beautifulsoup4==4.12.3 \
    lxml==5.3.0 \
    openai==1.42.0 \
    httpx==0.27.0 \
    streamlit==1.38.0 \
    pyngrok \
    tqdm==4.66.5 \
    ragas==0.1.19 \
    datasets==2.21.0

print("✅ All packages installed.")
print("👉 Restart the runtime now: Runtime → Restart session")
print("   Then run from Cell 2 onwards — do NOT re-run Cell 1.")


## Cell 2 — Credentials

Add these in Colab's **🔑 Secrets panel** (left sidebar). Toggle **Notebook access ON** for each one before running this cell.

| Secret name | Value |
|---|---|
| `AZURE_OPENAI_KEY` | Your Azure API key |
| `AZURE_OPENAI_ENDPOINT` | `https://YOUR-RESOURCE.openai.azure.com/` |
| `AZURE_OPENAI_DEPLOYMENT` | Your deployment name, e.g. `gpt-4o-mini` |

To use HuggingFace (free, no credits), set `LLM_PROVIDER = 'huggingface'`.

In [ ]:
import os
from google.colab import userdata

LLM_PROVIDER = 'azure_openai'   # or 'huggingface' for free option

if LLM_PROVIDER == 'azure_openai':
    os.environ['LLM_PROVIDER']                 = 'azure_openai'
    os.environ['AZURE_OPENAI_API_KEY']         = userdata.get('AZURE_OPENAI_KEY')
    os.environ['AZURE_OPENAI_ENDPOINT']        = userdata.get('AZURE_OPENAI_ENDPOINT')
    os.environ['AZURE_OPENAI_DEPLOYMENT_NAME'] = userdata.get('AZURE_OPENAI_DEPLOYMENT')
    os.environ['AZURE_OPENAI_API_VERSION']     = '2025-01-01-preview'
    print(f"Azure OpenAI — deployment: {userdata.get('AZURE_OPENAI_DEPLOYMENT')}")
else:
    os.environ['LLM_PROVIDER'] = 'huggingface'
    os.environ['HF_LLM_MODEL'] = 'mistralai/Mistral-7B-Instruct-v0.3'
    print('HuggingFace (free) — Mistral-7B')

os.environ['EMBEDDING_PROVIDER'] = 'huggingface'
os.environ['HF_EMBEDDING_MODEL'] = 'BAAI/bge-small-en-v1.5'
os.environ['VECTOR_STORE_TYPE']  = 'chroma'
os.environ['CHROMA_PERSIST_DIR'] = './data/chroma_db'
os.environ['RETRIEVAL_K']        = '8'
os.environ['LLM_TEMPERATURE']    = '0.1'
os.environ['LLM_MAX_TOKENS']     = '1024'

## Cell 3 — Setup

Creates `src/` and `data/` directories and adds `src/` to the Python path. **Must be run after every runtime restart** before any imports.

In [ ]:
import os, sys

os.makedirs('src',  exist_ok=True)
os.makedirs('data', exist_ok=True)

if 'src' not in sys.path:
    sys.path.insert(0, 'src')

print('src/ and data/ ready.')
print('src/ added to Python path.')


## Cells 4–8 — Write Source Modules

Each cell writes one Python module to `src/` using `%%writefile`. **Re-run these after every runtime restart.** They are quick — no downloads.

### Cell 4 — scraper.py
Web crawler (your `beautifulsoup_fixed.ipynb` logic): correct seed URLs, single-fetch, `>10` char filter, `%20` cleanup.

In [ ]:
%%writefile src/scraper.py
import time
import json
import logging
import os
from urllib.parse import urljoin, urlparse
from dataclasses import dataclass, field, asdict
from typing import Optional

import requests
from bs4 import BeautifulSoup

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

BASE_URL    = "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/"
DOMAIN      = "datascience.uchicago.edu"
PATH_PREFIX = "/education/masters-programs/ms-in-applied-data-science/"

CRAWL_DELAY   = 1.2
CHUNK_SIZE    = 150
CHUNK_OVERLAP = 30
OUT_DIR       = "data"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Connection":      "keep-alive",
}

SEED_URLS = [
    BASE_URL,
    BASE_URL + "in-person-program/",
    BASE_URL + "online-program/",
    BASE_URL + "course-progressions/",
    BASE_URL + "how-to-apply/",
    BASE_URL + "instructors-staff/",
    BASE_URL + "tuition-fees-aid/",
    BASE_URL + "career-outcomes/",
    BASE_URL + "faqs/",
    BASE_URL + "capstone-projects/",
    BASE_URL + "events-deadlines/",
    BASE_URL + "our-students/",
]


@dataclass
class PageChunk:
    url:         str
    page_title:  str
    section:     str
    chunk_index: int
    text:        str

@dataclass
class ScrapedPage:
    url:        str
    page_title: str
    section:    str
    raw_text:   str
    chunks:     list = field(default_factory=list)


def section_from_url(url: str) -> str:
    path = urlparse(url).path.rstrip("/")
    last = path.split("/")[-1]
    if not last or "ms-in-applied-data-science" in last:
        return "Overview"
    return last.replace("-", " ").title()


def html_to_clean_text(soup: BeautifulSoup) -> str:
    for tag in soup(["script", "style", "nav", "footer", "header",
                     "aside", "form", "noscript", "iframe", "button"]):
        tag.decompose()

    noise_patterns = [
        "nav", "menu", "footer", "header", "sidebar", "cookie",
        "banner", "breadcrumb", "social", "share", "alert",
        "announcement", "skip-link", "site-header", "site-footer",
        "wp-admin-bar",
    ]
    for pattern in noise_patterns:
        for el in soup.find_all(class_=lambda c: c and pattern in " ".join(c).lower()):
            el.decompose()
        for el in soup.find_all(id=lambda i: i and pattern in i.lower()):
            el.decompose()

    main_content = (
        soup.find("main")
        or soup.find("article")
        or soup.find(id="content")
        or soup.find(id="main-content")
        or soup.find(class_="entry-content")
        or soup.find(class_="page-content")
        or soup.find("body")
        or soup
    )

    raw = main_content.get_text(separator="\n")

    lines = []
    for line in raw.splitlines():
        line = line.strip()
        if len(line) > 10:
            lines.append(line)

    return "\n\n".join(lines)


def chunk_text(text: str, url: str, title: str, section: str) -> list:
    """
    Split text into overlapping chunks.
    Each chunk is prefixed with its page title and section so that
    embeddings are anchored to the topic — preventing off-topic
    content at chunk boundaries from dominating the embedding.
    """
    words = text.split()
    if not words:
        return []

    # Build a short header that gets prepended to every chunk
    # This ensures every chunk's embedding reflects its true topic
    header = f"Page: {title}. Section: {section}. "

    chunks, i, idx = [], 0, 0
    while i < len(words):
        chunk_words = words[i: i + CHUNK_SIZE]
        chunk_str = header + " ".join(chunk_words)
        chunks.append(PageChunk(
            url=url, page_title=title, section=section,
            chunk_index=idx, text=chunk_str
        ))
        idx += 1
        i += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks


def fetch_page(url: str, session: requests.Session) -> tuple:
    try:
        response = session.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
    except requests.RequestException as e:
        log.warning(f"  x Failed: {url}  ({type(e).__name__}: {e})")
        return None, None

    content_type = response.headers.get("Content-Type", "")
    if "text/html" not in content_type:
        log.info(f"  Skipped (not HTML): {url}")
        return None, None

    raw_html = response.text
    soup = BeautifulSoup(raw_html, "lxml")

    title_tag = soup.find("title")
    title = title_tag.get_text(strip=True) if title_tag else section_from_url(url)
    title = title.split("|")[0].strip() if "|" in title else title

    section  = section_from_url(url)
    raw_text = html_to_clean_text(soup)

    if len(raw_text) < 100:
        log.info(f"  Skipped (too little content): {url}")
        return None, raw_html

    chunks = chunk_text(raw_text, url, title, section)
    log.info(f"  OK [{section:20s}]  {len(raw_text):6,} chars  ->  {len(chunks):3d} chunks")

    return ScrapedPage(url=url, page_title=title, section=section,
                       raw_text=raw_text, chunks=chunks), raw_html


def discover_links(html: str, base_url: str) -> list:
    soup = BeautifulSoup(html, "lxml")
    found = []
    skip_ext = {".pdf", ".docx", ".xlsx", ".zip", ".png", ".jpg", ".gif"}

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if href.startswith("#"):
            continue

        full_url = urljoin(base_url, href)
        parsed   = urlparse(full_url)

        if parsed.netloc != DOMAIN:
            continue
        if not parsed.path.startswith(PATH_PREFIX):
            continue
        if any(parsed.path.lower().endswith(ext) for ext in skip_ext):
            continue

        clean_path = parsed.path.replace("%20", "").rstrip("/") + "/"
        clean = parsed._replace(path=clean_path, fragment="", query="").geturl()

        if clean not in found:
            found.append(clean)

    return found


def crawl() -> list:
    session  = requests.Session()
    visited  = set()
    queue    = list(SEED_URLS)
    pages    = []

    log.info(f"Starting crawl | {len(SEED_URLS)} seed URLs | domain: {DOMAIN}")
    log.info("-" * 70)

    while queue:
        url        = queue.pop(0)
        normalized = url.rstrip("/") + "/"

        if normalized in visited:
            continue
        visited.add(normalized)

        log.info(f"Fetching ({len(visited)}/{len(visited)+len(queue)}): {url}")

        page, raw_html = fetch_page(url, session)

        if page is not None:
            pages.append(page)

        if raw_html:
            new_links = discover_links(raw_html, url)
            added = 0
            for link in new_links:
                norm = link.rstrip("/") + "/"
                if norm not in visited and link not in queue:
                    queue.append(link)
                    added += 1
            if added:
                log.info(f"    -> Discovered {added} new link(s)")

        time.sleep(CRAWL_DELAY)

    log.info("-" * 70)
    log.info(f"Done: {len(pages)} pages, {sum(len(p.chunks) for p in pages)} chunks")
    return pages


def save_results(pages: list) -> None:
    os.makedirs(OUT_DIR, exist_ok=True)

    pages_path = os.path.join(OUT_DIR, "scraped_pages.json")
    with open(pages_path, "w", encoding="utf-8") as f:
        json.dump([asdict(p) for p in pages], f, indent=2, ensure_ascii=False)
    log.info(f"Saved -> {pages_path}")

    all_chunks = [asdict(c) for p in pages for c in p.chunks]
    chunks_path = os.path.join(OUT_DIR, "all_chunks.json")
    with open(chunks_path, "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, indent=2, ensure_ascii=False)
    log.info(f"Saved {len(all_chunks)} chunks -> {chunks_path}")

    txt_path = os.path.join(OUT_DIR, "raw_text.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        for page in pages:
            f.write(f"\n{'='*70}\n")
            f.write(f"SECTION : {page.section}\n")
            f.write(f"URL     : {page.url}\n")
            f.write(f"TITLE   : {page.page_title}\n")
            f.write(f"{'='*70}\n\n")
            f.write(page.raw_text)
            f.write("\n")
    log.info(f"Saved readable text -> {txt_path}")


def print_summary(pages: list) -> None:
    print("\n" + "=" * 65)
    print(f"{'SECTION':<25} {'CHUNKS':>6}  {'CHARS':>8}")
    print("-" * 65)
    total_chunks = total_chars = 0
    for page in pages:
        c, ch = len(page.chunks), len(page.raw_text)
        total_chunks += c; total_chars += ch
        print(f"{page.section:<25} {c:>6}  {ch:>8,}")
    print("-" * 65)
    print(f"{'TOTAL':<25} {total_chunks:>6}  {total_chars:>8,}")
    print("=" * 65)


### Cell 5 — embedder.py
Loads chunks → LangChain Documents, manages HuggingFace / Azure embeddings, builds and loads ChromaDB / FAISS vector stores.

In [ ]:
%%writefile src/embedder.py
import os
import json
import logging
from langchain.schema import Document

log = logging.getLogger(__name__)

EMBEDDING_PROVIDER = os.getenv("EMBEDDING_PROVIDER", "huggingface")
HF_EMBEDDING_MODEL = os.getenv("HF_EMBEDDING_MODEL", "BAAI/bge-small-en-v1.5")
VECTOR_STORE_TYPE  = os.getenv("VECTOR_STORE_TYPE", "chroma")
CHROMA_PERSIST_DIR = os.getenv("CHROMA_PERSIST_DIR", "./data/chroma_db")
AZURE_API_KEY      = os.getenv("AZURE_OPENAI_API_KEY", "")
AZURE_ENDPOINT     = os.getenv("AZURE_OPENAI_ENDPOINT", "")
AZURE_EMB_DEPLOY   = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-small")
AZURE_API_VERSION  = os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")


def chunks_to_documents(chunks_path: str = "data/all_chunks.json") -> list:
    """Convert scraped chunks JSON -> LangChain Documents."""
    with open(chunks_path, "r", encoding="utf-8") as f:
        chunks = json.load(f)
    docs = []
    for c in chunks:
        if not c.get("text", "").strip():
            continue
        docs.append(Document(
            page_content=c["text"],
            metadata={
                "source":      c["url"],
                "page_title":  c.get("page_title", ""),
                "section":     c.get("section", ""),
                "chunk_index": c.get("chunk_index", 0),
            }
        ))
    log.info("Loaded %d documents from %s", len(docs), chunks_path)
    return docs


def get_embeddings(provider: str = EMBEDDING_PROVIDER):
    provider = provider.lower()
    if provider == "huggingface":
        from langchain_huggingface import HuggingFaceEmbeddings
        log.info("Loading HuggingFace embeddings: %s", HF_EMBEDDING_MODEL)
        return HuggingFaceEmbeddings(
            model_name=HF_EMBEDDING_MODEL,
            model_kwargs={"device": "cpu"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif provider == "azure_openai":
        from langchain_openai import AzureOpenAIEmbeddings
        log.info("Loading Azure OpenAI embeddings: %s", AZURE_EMB_DEPLOY)
        return AzureOpenAIEmbeddings(
            azure_deployment=AZURE_EMB_DEPLOY,
            azure_endpoint=AZURE_ENDPOINT,
            api_key=AZURE_API_KEY,
            api_version=AZURE_API_VERSION,
        )
    elif provider == "openai":
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model="text-embedding-3-small")
    else:
        raise ValueError(f"Unknown embedding provider: {provider!r}")


class VectorStoreManager:
    def __init__(self, store_type=VECTOR_STORE_TYPE, embeddings=None,
                 persist_dir=CHROMA_PERSIST_DIR):
        self.store_type  = store_type.lower()
        self.persist_dir = persist_dir
        self.embeddings  = embeddings or get_embeddings()
        self._store      = None

    def build(self, documents: list) -> None:
        log.info("Building %s vector store with %d documents...",
                 self.store_type, len(documents))
        if self.store_type == "chroma":
            from langchain_community.vectorstores import Chroma
            os.makedirs(self.persist_dir, exist_ok=True)
            self._store = Chroma.from_documents(
                documents=documents,
                embedding=self.embeddings,
                persist_directory=self.persist_dir,
                collection_name="msads_rag",
            )
        elif self.store_type == "faiss":
            from langchain_community.vectorstores import FAISS
            self._store = FAISS.from_documents(documents, self.embeddings)
        else:
            raise ValueError(f"Unknown store type: {self.store_type!r}")
        log.info("Vector store built successfully.")

    def load(self) -> None:
        log.info("Loading %s vector store from %s", self.store_type, self.persist_dir)
        if self.store_type == "chroma":
            from langchain_community.vectorstores import Chroma
            self._store = Chroma(
                persist_directory=self.persist_dir,
                embedding_function=self.embeddings,
                collection_name="msads_rag",
            )
        elif self.store_type == "faiss":
            from langchain_community.vectorstores import FAISS
            self._store = FAISS.load_local(
                self.persist_dir, self.embeddings,
                allow_dangerous_deserialization=True,
            )

    def retriever(self, k: int = 5):
        if self._store is None:
            raise RuntimeError("Call build() or load() first.")
        return self._store.as_retriever(
            search_type="similarity", search_kwargs={"k": k}
        )

    def similarity_search(self, query: str, k: int = 5):
        if self._store is None:
            raise RuntimeError("Call build() or load() first.")
        return self._store.similarity_search(query, k=k)

    def similarity_search_with_score(self, query: str, k: int = 5):
        if self._store is None:
            raise RuntimeError("Call build() or load() first.")
        return self._store.similarity_search_with_score(query, k=k)


def build_vector_store(documents, provider=EMBEDDING_PROVIDER,
                       store_type=VECTOR_STORE_TYPE) -> VectorStoreManager:
    vsm = VectorStoreManager(store_type=store_type,
                              embeddings=get_embeddings(provider))
    vsm.build(documents)
    return vsm


def load_vector_store(provider=EMBEDDING_PROVIDER,
                      store_type=VECTOR_STORE_TYPE,
                      persist_dir=CHROMA_PERSIST_DIR) -> VectorStoreManager:
    vsm = VectorStoreManager(store_type=store_type,
                              embeddings=get_embeddings(provider),
                              persist_dir=persist_dir)
    vsm.load()
    return vsm


### Cell 6 — llm_factory.py
Returns the right LangChain LLM based on `LLM_PROVIDER`: Azure OpenAI, OpenAI, or HuggingFace.

In [ ]:
%%writefile src/llm_factory.py
import os
import logging

log = logging.getLogger(__name__)

LLM_PROVIDER   = os.getenv("LLM_PROVIDER", "azure_openai")
AZURE_API_KEY  = os.getenv("AZURE_OPENAI_API_KEY", "")
AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "")
AZURE_DEPLOY   = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-4o-mini")
AZURE_VERSION  = os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")
HF_LLM_MODEL   = os.getenv("HF_LLM_MODEL", "mistralai/Mistral-7B-Instruct-v0.3")
HF_API_KEY     = os.getenv("HUGGINGFACE_API_KEY", "")
LLM_TEMP       = float(os.getenv("LLM_TEMPERATURE", "0.1"))
LLM_MAX_TOKENS = int(os.getenv("LLM_MAX_TOKENS", "1024"))


def get_llm(provider: str = LLM_PROVIDER):
    provider = provider.lower()

    if provider == "azure_openai":
        log.info("Using Azure OpenAI LLM: %s", AZURE_DEPLOY)
        from langchain_openai import AzureChatOpenAI
        return AzureChatOpenAI(
            azure_deployment=AZURE_DEPLOY,
            azure_endpoint=AZURE_ENDPOINT,
            api_key=AZURE_API_KEY,
            api_version=AZURE_VERSION,
            temperature=LLM_TEMP,
            max_tokens=LLM_MAX_TOKENS,
        )

    elif provider == "openai":
        log.info("Using OpenAI LLM: gpt-4o-mini")
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(
            model="gpt-4o-mini",
            temperature=LLM_TEMP,
            max_tokens=LLM_MAX_TOKENS,
        )

    elif provider == "huggingface":
        log.info("Using HuggingFace LLM: %s", HF_LLM_MODEL)
        from langchain_huggingface import HuggingFaceEndpoint
        return HuggingFaceEndpoint(
            repo_id=HF_LLM_MODEL,
            huggingfacehub_api_token=HF_API_KEY or None,
            temperature=LLM_TEMP,
            max_new_tokens=LLM_MAX_TOKENS,
            task="text-generation",
        )

    else:
        raise ValueError(
            f"Unknown LLM provider: {provider!r}. "
            "Choose 'azure_openai', 'openai', or 'huggingface'."
        )


### Cell 7 — rag_chain.py
Full RAG pipeline: question condensation, retrieval, grounded answer generation, PII redaction, source citation, conversational memory.

In [ ]:
%%writefile src/rag_chain.py
import os
import re
import logging
from langchain.prompts import ChatPromptTemplate, PromptTemplate, MessagesPlaceholder
from langchain.schema.output_parser import StrOutputParser
from langchain_core.messages import AIMessage, HumanMessage

log = logging.getLogger(__name__)

RETRIEVAL_K = int(os.getenv("RETRIEVAL_K", "8"))

SYSTEM_PROMPT = (
    "You are an expert assistant for the MS in Applied Data Science (MS-ADS) program "
    "at the University of Chicago. Help prospective students, current students, and alumni.\n\n"
    "RULES:\n"
    "1. Read ALL provided context chunks carefully. Answers are often spread across "
    "multiple chunks — synthesise everything relevant into one complete answer.\n"
    "2. If context has partial info, use it and suggest visiting https://datascience.uchicago.edu\n"
    "3. Only say you lack information if context is completely irrelevant to the question.\n"
    "4. Always cite source URL(s) at the end.\n"
    "5. Use bullet points for lists. Be concise but thorough.\n"
    "6. Never invent course names, deadlines, tuition figures, or faculty names.\n\n"
    "CONTEXT:\n{context}"
)

CONDENSE_PROMPT = (
    "Given the conversation history and a follow-up question, rewrite the follow-up "
    "as a complete standalone question. Do not answer — only rewrite.\n\n"
    "Conversation history:\n{chat_history}\n\n"
    "Follow-up question: {question}\n"
    "Standalone question:"
)

PII_PATTERNS = [
    (re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),                             "[SSN REDACTED]"),
    (re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"), "[EMAIL REDACTED]"),
    (re.compile(r"\(\d{3}\)\s?\d{3}-\d{4}|\b\d{10}\b"),            "[PHONE REDACTED]"),
]

OFF_TOPIC_KEYWORDS = [
    "stock price", "weather forecast", "recipe", "movie review",
    "bitcoin", "dating", "song lyrics", "sports score",
]


def redact_pii(text: str) -> str:
    for pattern, replacement in PII_PATTERNS:
        text = pattern.sub(replacement, text)
    return text


def is_off_topic(query: str) -> bool:
    return any(kw in query.lower() for kw in OFF_TOPIC_KEYWORDS)


def format_docs(docs) -> str:
    parts = []
    for i, doc in enumerate(docs, 1):
        meta = doc.metadata
        parts.append(
            f"[Chunk {i}]\n"
            f"Section: {meta.get('section', '')}\n"
            f"Source:  {meta.get('source', '')}\n\n"
            f"{doc.page_content}"
        )
    return "\n\n" + ("─" * 60 + "\n\n").join(parts)


def extract_sources(docs) -> list:
    seen, urls = set(), []
    for doc in docs:
        url = doc.metadata.get("source", "")
        if url and url not in seen:
            seen.add(url)
            urls.append(url)
    return urls


class RAGChain:
    def __init__(self, retriever, llm):
        self.retriever    = retriever
        self.llm          = llm
        self.chat_history = []

        self.condense_chain = (
            PromptTemplate.from_template(CONDENSE_PROMPT)
            | self.llm
            | StrOutputParser()
        )

        self.answer_chain = (
            ChatPromptTemplate.from_messages([
                ("system", SYSTEM_PROMPT),
                MessagesPlaceholder(variable_name="chat_history"),
                ("human", "{question}"),
            ])
            | self.llm
            | StrOutputParser()
        )

    def _condense_question(self, question: str) -> str:
        if not self.chat_history:
            return question
        history_str = "\n".join(
            f"Human: {m.content}" if isinstance(m, HumanMessage)
            else f"Assistant: {m.content}"
            for m in self.chat_history
        )
        try:
            return self.condense_chain.invoke({
                "chat_history": history_str,
                "question":     question,
            })
        except Exception:
            return question

    def ask(self, question: str, k: int = RETRIEVAL_K) -> dict:
        if is_off_topic(question):
            return {
                "answer": (
                    "I specialise in the MS in Applied Data Science program at UChicago. "
                    "Please ask about courses, admissions, faculty, tuition, or career outcomes."
                ),
                "sources": [], "docs": [], "standalone_question": question,
            }

        standalone_q = self._condense_question(question)
        docs         = self.retriever.invoke(standalone_q)

        raw_answer = self.answer_chain.invoke({
            "context":      format_docs(docs),
            "chat_history": self.chat_history,
            "question":     standalone_q,
        })

        answer  = redact_pii(raw_answer)
        sources = extract_sources(docs)

        if sources and "**Sources:**" not in answer:
            answer += "\n\n**Sources:**\n" + "\n".join(f"- {s}" for s in sources)

        self.chat_history.append(HumanMessage(content=question))
        self.chat_history.append(AIMessage(content=answer))
        if len(self.chat_history) > 20:
            self.chat_history = self.chat_history[-20:]

        return {
            "answer":              answer,
            "sources":             sources,
            "docs":                docs,
            "standalone_question": standalone_q,
        }

    def reset(self) -> None:
        self.chat_history = []


def build_rag_chain(vsm, llm) -> RAGChain:
    return RAGChain(retriever=vsm.retriever(k=RETRIEVAL_K), llm=llm)


### Cell 8 — evaluator.py
Heuristic evaluation (no API cost) + optional RAGAS evaluation (LLM-as-judge). 10 curated Q&A pairs covering all 13 scraped sections.

In [ ]:
%%writefile src/evaluator.py
import os
import json
import logging
from datetime import datetime

log = logging.getLogger(__name__)

DEFAULT_TEST_SET = [
    {
        "question": "What are the core courses in the MS in Applied Data Science program?",
        "ground_truth": (
            "The core courses include Machine Learning, Data Engineering Platforms, "
            "Statistical Inference, and Applied Data Science."
        ),
    },
    {
        "question": "What are the admission requirements for the MS-ADS program?",
        "ground_truth": (
            "Applicants need a bachelor's degree in a related field with coursework in "
            "programming, statistics, and mathematics, plus a personal statement, "
            "letters of recommendation, and a resume."
        ),
    },
    {
        "question": "Can I study the MS-ADS program online?",
        "ground_truth": (
            "Yes, the MS in Applied Data Science is available in both in-person "
            "and online formats."
        ),
    },
    {
        "question": "What is the capstone project in the MS-ADS program?",
        "ground_truth": (
            "The capstone project is a key component where students work on real-world "
            "data science problems, applying their skills to develop data-driven solutions."
        ),
    },
    {
        "question": "What career outcomes do MS-ADS graduates typically achieve?",
        "ground_truth": (
            "Graduates work as data scientists, machine learning engineers, "
            "data analysts, and AI researchers across various industries."
        ),
    },
    {
        "question": "How long does the MS in Applied Data Science program take?",
        "ground_truth": (
            "The program is typically completed in about one year of full-time study."
        ),
    },
    {
        "question": "What is the tuition for the MS-ADS program?",
        "ground_truth": (
            "Tuition and fee details are listed on the UChicago MS-ADS tuition page. "
            "Financial aid and scholarships may be available."
        ),
    },
    {
        "question": "Who are the instructors in the MS-ADS program?",
        "ground_truth": (
            "The program has faculty with expertise in machine learning, statistics, "
            "data engineering, and applied data science from the University of Chicago."
        ),
    },
    {
        "question": "What is the difference between the online and in-person MS-ADS programs?",
        "ground_truth": (
            "Both programs share the same curriculum and degree. The in-person program "
            "is held in Chicago, while the online program allows remote participation "
            "with the same coursework and faculty."
        ),
    },
    {
        "question": "What are the application deadlines for the MS-ADS program?",
        "ground_truth": (
            "Application deadlines vary by quarter. Check the events and deadlines page "
            "on the MS-ADS website for current dates."
        ),
    },
]


def simple_evaluate(question: str, answer: str, retrieved_docs: list) -> dict:
    """
    Fast heuristic evaluation — no LLM calls needed.
    Returns scores in [0, 1] for each dimension.
    """
    answer_lower = answer.lower()

    # 1. Has a real answer (not a fallback)
    fallback_phrases = [
        "don't have that information",
        "i don't have",
        "not in my knowledge base",
        "please contact",
    ]
    has_answer = (
        len(answer.strip()) > 30
        and not any(p in answer_lower for p in fallback_phrases)
    )

    # 2. Retrieved enough context
    context_score = min(len(retrieved_docs) / 5, 1.0)

    # 3. Keyword overlap between question and answer
    stop_words = {
        "what", "is", "the", "are", "a", "an", "in", "of", "for",
        "how", "can", "i", "do", "does", "tell", "me", "about",
        "program", "ms", "ads", "uchicago",
    }
    q_keywords = set(question.lower().split()) - stop_words
    a_words    = set(answer_lower.split())
    overlap    = q_keywords & a_words
    relevance  = min(len(overlap) / max(len(q_keywords), 1), 1.0)

    # 4. Faithfulness proxy: penalise hedging/uncertainty language
    hedging_phrases = [
        "i think", "i believe", "probably", "might be",
        "not certain", "i'm not sure",
    ]
    has_hedging  = any(p in answer_lower for p in hedging_phrases)
    faithfulness = 0.7 if has_hedging else 1.0

    # 5. Has source citation
    has_sources = "sources:" in answer_lower or "source:" in answer_lower
    citation_score = 1.0 if has_sources else 0.5

    composite = (
        float(has_answer) + context_score + relevance + faithfulness + citation_score
    ) / 5

    return {
        "has_answer":     float(has_answer),
        "context_score":  round(context_score, 4),
        "relevance":      round(relevance, 4),
        "faithfulness":   round(faithfulness, 4),
        "citation_score": round(citation_score, 4),
        "composite":      round(composite, 4),
    }


def run_simple_evaluation(
    rag_chain,
    test_set: list = None,
    output_path: str = "data/eval_results.json",
    verbose: bool = True,
) -> dict:
    """
    Runs heuristic evaluation on the test set.
    No LLM API calls — works offline/free.
    """
    if test_set is None:
        test_set = DEFAULT_TEST_SET

    log.info("Running simple evaluation on %d questions...", len(test_set))

    records = []
    metric_keys = ["has_answer", "context_score", "relevance",
                   "faithfulness", "citation_score", "composite"]
    totals = {k: 0.0 for k in metric_keys}

    for i, item in enumerate(test_set, 1):
        q  = item["question"]
        gt = item.get("ground_truth", "")
        if verbose:
            print(f"  [{i}/{len(test_set)}] {q[:60]}...")

        try:
            result = rag_chain.ask(q)
            ans    = result["answer"]
            docs   = result["docs"]
            scores = simple_evaluate(q, ans, docs)
        except Exception as exc:
            log.warning("Error on question %r: %s", q, exc)
            ans    = ""
            docs   = []
            scores = {k: 0.0 for k in metric_keys}

        record = {
            "question":     q,
            "ground_truth": gt,
            "answer":       ans,
            "num_docs_retrieved": len(docs),
            **scores,
        }
        records.append(record)
        for k in metric_keys:
            totals[k] += scores.get(k, 0.0)

    n        = len(test_set)
    averages = {f"avg_{k}": round(v / n, 4) for k, v in totals.items()}

    output = {
        "evaluated_at":  datetime.now().isoformat(),
        "num_questions": n,
        "averages":      averages,
        "per_question":  records,
    }

    os.makedirs(os.path.dirname(output_path) if os.path.dirname(output_path) else ".", exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as fh:
        json.dump(output, fh, indent=2, ensure_ascii=False)

    log.info("Evaluation complete. Results saved to %s", output_path)
    return output


def run_ragas_evaluation(
    rag_chain,
    test_set: list = None,
    output_path: str = "data/ragas_eval_results.json",
) -> dict:
    """
    Full RAGAS evaluation (LLM-as-judge).
    Requires: pip install ragas datasets
    Uses API credits — run after simple evaluation passes.
    """
    try:
        from ragas import evaluate
        from ragas.metrics import (
            faithfulness,
            answer_relevancy,
            context_precision,
            context_recall,
        )
        from datasets import Dataset
    except ImportError:
        raise ImportError(
            "RAGAS not installed. Run: pip install ragas datasets"
        )

    if test_set is None:
        test_set = DEFAULT_TEST_SET

    log.info("Running RAGAS evaluation on %d questions...", len(test_set))

    questions, answers, contexts, ground_truths = [], [], [], []

    for item in test_set:
        q  = item["question"]
        gt = item.get("ground_truth", "")
        try:
            result  = rag_chain.ask(q)
            ans     = result["answer"]
            ctx_lst = [d.page_content for d in result["docs"]]
        except Exception as exc:
            log.warning("Error on %r: %s", q, exc)
            ans, ctx_lst = "", []

        questions.append(q)
        answers.append(ans)
        contexts.append(ctx_lst)
        ground_truths.append(gt)

    dataset = Dataset.from_dict({
        "question":     questions,
        "answer":       answers,
        "contexts":     contexts,
        "ground_truth": ground_truths,
    })

    result  = evaluate(
        dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    )
    metrics = dict(result)
    metrics["evaluated_at"]  = datetime.now().isoformat()
    metrics["num_questions"] = len(test_set)

    os.makedirs(os.path.dirname(output_path) if os.path.dirname(output_path) else ".", exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as fh:
        json.dump(metrics, fh, indent=2)

    log.info("RAGAS evaluation complete -> %s", output_path)
    return metrics


## Cell 9 — Scrape the MS-ADS Website

Crawls 13 pages live. Expected: **13 pages, 70 chunks, ~193K chars**. Takes ~25 seconds. Saves to `data/scraped_pages.json`, `data/all_chunks.json`, `data/raw_text.txt`.

**Skip this cell** if you already have `data/all_chunks.json` from a previous run (e.g. saved to Google Drive).

In [ ]:
from scraper import crawl, save_results, print_summary

print('🎓 UChicago MS-ADS Web Scraper')

pages = crawl()

if not pages:
    print('\nNo pages scraped. Check internet connection.')
else:
    print_summary(pages)
    save_results(pages)
    print(f'\n{len(pages)} pages, {sum(len(p.chunks) for p in pages)} chunks.')


## Cell 10 — Inspect Scraped Data

Review chunk counts, word counts, and sample text per section before embedding.

In [ ]:
import json
import pandas as pd

with open('data/all_chunks.json') as f:
    all_chunks = json.load(f)

df = pd.DataFrame(all_chunks)
df['word_count'] = df['text'].str.split().str.len()

print(f'Total chunks : {len(df)}')
print(f'Total words  : {df["word_count"].sum():,}')
print()
print('Chunks per section:')
print(df.groupby('section')['chunk_index'].count().rename('chunks').to_string())
print()
print('Word count per chunk (stats):')
print(df['word_count'].describe().round(1).to_string())
print()
print('Sample chunk from each section:')
print('=' * 65)
for section, group in df.groupby('section'):
    row = group.iloc[0]
    print(f'\n[{section}]')
    print(f'URL   : {row["url"]}')
    print(f'Words : {row["word_count"]}')
    print(f'Text  : {row["text"][:200]}...')


## Cell 11 — Embed Chunks & Build Vector Store

Downloads **BAAI/bge-small-en-v1.5** (~133MB, first run only), embeds all chunks, and persists them in ChromaDB.

**Important:** In addition to the scraped chunks, this cell adds **3 curated course summary chunks** that list all core and elective course names in a clean, focused format. This is necessary because the scraper's chunks break the course list across many 150-word chunks — each containing only fragments of course names buried in descriptions. The curated chunks give the embedding model a clean target to match course-name queries against.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from embedder import chunks_to_documents, build_vector_store
from langchain.schema import Document

print('Loading chunks from data/all_chunks.json...')
documents = chunks_to_documents('data/all_chunks.json')
print(f'  {len(documents)} documents from scraper')

# ── Add curated course summary chunks ────────────────────────────────────────
# Problem: the scraper's 150-word chunks break the course list across many
# chunks. Each individual chunk has poor embedding similarity to "core courses"
# because it contains only ONE course name buried in a paragraph of description.
#
# Fix: add clean summary chunks that list ALL course names together.
# These are taken directly from the scraped raw_text.txt and compressed
# into focused chunks the embedder can match strongly against course queries.

BASE_URL = "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/"

curated_chunks = [
    Document(
        page_content=(
            "Core Courses in the MS in Applied Data Science program (6 required):\n"
            "1. Statistical Models for Data Science\n"
            "2. Machine Learning I\n"
            "3. Machine Learning II\n"
            "4. Data Engineering Platforms for Analytics (or Big Data and Cloud Computing)\n"
            "5. Leadership and Consulting for Data Science\n"
            "6. Time Series Analysis and Forecasting\n\n"
            "All students complete these 6 core courses. They build theoretical data science "
            "knowledge and teach students to apply methods to real-world business problems."
        ),
        metadata={
            "source":     BASE_URL + "in-person-program/",
            "section":    "Core Courses",
            "page_title": "MS-ADS Core Courses",
            "chunk_index": 0,
        }
    ),
    Document(
        page_content=(
            "Sample Elective Courses in the MS in Applied Data Science program (choose 4):\n"
            "- Advanced Computer Vision with Deep Learning\n"
            "- Advanced Machine Learning and Artificial Intelligence\n"
            "- Bayesian Machine Learning with Generative AI Applications\n"
            "- Data Science for Algorithmic Marketing\n"
            "- Data Visualization Techniques\n"
            "- Digital Marketing Analytics in Theory and Practice\n"
            "- Quantitative Finance: Methods and Applications\n"
            "- Data Science for Healthcare\n"
            "- Machine Learning Operations\n"
            "- Next-Gen NLP: LLM and Agentic AI in Practice\n"
            "- Real Time Intelligent Systems\n"
            "- Deep Reinforcement Learning\n"
            "- Supply Chain Optimization\n\n"
            "Students in the 12-course track complete 4 electives. "
            "Electives evolve with the data science landscape."
        ),
        metadata={
            "source":     BASE_URL + "online-program/",
            "section":    "Elective Courses",
            "page_title": "MS-ADS Elective Courses",
            "chunk_index": 0,
        }
    ),
    Document(
        page_content=(
            "MS in Applied Data Science — Full Curriculum Overview:\n\n"
            "The 12-course program consists of:\n"
            "- 6 Core Courses (required for all students):\n"
            "  1. Statistical Models for Data Science\n"
            "  2. Machine Learning I\n"
            "  3. Machine Learning II\n"
            "  4. Data Engineering Platforms for Analytics\n"
            "  5. Leadership and Consulting for Data Science\n"
            "  6. Time Series Analysis and Forecasting\n"
            "- 4 Elective Courses (chosen from a rotating list)\n"
            "- 2 Capstone Project courses (completed over two quarters)\n"
            "- Career Seminar (pass/fail, required each quarter)\n\n"
            "Optional Foundational courses (free, 0 units) include: "
            "Python for Data Science, R for Data Science, "
            "Introduction to Statistical Concepts, "
            "Advanced Linear Algebra for Machine Learning."
        ),
        metadata={
            "source":     BASE_URL + "course-progressions/",
            "section":    "Curriculum Overview",
            "page_title": "MS-ADS Curriculum Overview",
            "chunk_index": 0,
        }
    ),
]

documents = documents + curated_chunks
print(f'  {len(curated_chunks)} curated course summary chunks added')
print(f'  {len(documents)} total documents')

print('\nBuilding vector store...')
vsm = build_vector_store(
    documents  = documents,
    provider   = os.environ['EMBEDDING_PROVIDER'],
    store_type = os.environ['VECTOR_STORE_TYPE'],
)

print('\n✅ Vector store built and saved to data/chroma_db/')
print('   Curated course chunks ensure course name queries retrieve accurate results.')


## Cell 12 — Test Retrieval

Run four queries directly against the vector store to verify it is working before connecting the LLM.

In [ ]:
test_queries = [
    'What are the core courses?',
    'How do I apply to the program?',
    'Who are the instructors?',
    'What is the tuition fee?',
]

for query in test_queries:
    print(f'\n🔍 "{query}"')
    results = vsm.similarity_search(query, k=3)
    for i, r in enumerate(results, 1):
        m = r.metadata
        print(f'  [{i}] {m["section"]:25s} | {m["source"][-50:]}')
        print(f'       {r.page_content[:120].strip()}...')

print('\nRetrieval working correctly!')


## Cell 13 — Build RAG Chain

Connects the vector store retriever to the LLM. Prints full system config.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from llm_factory import get_llm
from rag_chain   import build_rag_chain

print('Initialising LLM...')
llm = get_llm(provider=os.environ['LLM_PROVIDER'])

print('Building RAG chain...')
chain = build_rag_chain(vsm, llm)

print(f'   LLM        : {os.environ["LLM_PROVIDER"]}')
print(f'   Model      : {os.environ.get("AZURE_OPENAI_DEPLOYMENT_NAME", os.environ.get("HF_LLM_MODEL", "-"))}')
print(f'   Embeddings : {os.environ["HF_EMBEDDING_MODEL"]}')
print(f'   Store      : {os.environ["VECTOR_STORE_TYPE"]} → {os.environ["CHROMA_PERSIST_DIR"]}')
print(f'   Retrieval  : top-{os.environ["RETRIEVAL_K"]} chunks per query')


## Cell 14 — Interactive Q&A

Five questions covering the main sections of the MS-ADS program.

In [ ]:
chain.reset()

questions = [
    'What are the core courses in the MS-ADS program?',
    'What are the admission requirements and how do I apply?',
    'What is the capstone project and when does it happen?',
    'What is the tuition for the online program?',
    'What career outcomes have past graduates achieved?',
]

for q in questions:
    print('\n' + '=' * 70)
    print(f'Q: {q}')
    print('-' * 70)
    result = chain.ask(q)
    print(result['answer'])
    print(f'\n[{len(result["docs"])} chunks retrieved from {len(result["sources"])} page(s)]')


## Cell 15 — Multi-Turn Conversation

Shows conversational memory. Vague follow-ups like *'Does it require work experience?'* are automatically rewritten as standalone questions so retrieval still works.

In [ ]:
chain.reset()

conversation = [
    'What are the admission requirements for the MS-ADS program?',
    'Does it require work experience?',
    'What about GRE or GMAT scores?',
    'How is the online program different from in-person?',
    'Do both formats lead to the same degree?',
]

for i, q in enumerate(conversation, 1):
    print(f'\n[Turn {i}]')
    print(f'Q: {q}')
    result = chain.ask(q)
    sq = result.get('standalone_question', q)
    if sq.strip().lower() != q.strip().lower():
        print(f'   → condensed to: "{sq}"')
    print(f'A: {result["answer"][:500]}')
    if len(result['answer']) > 500:
        print('   [...truncated for display...]')


## Cell 16 — Evaluation (Heuristic — No API Cost)

Tests the system on 10 curated Q&A pairs. No LLM calls — runs in seconds.

| Metric | What it measures |
|---|---|
| `has_answer` | Real answer given (not a fallback) |
| `context_score` | Enough chunks retrieved |
| `relevance` | Answer shares keywords with question |
| `faithfulness` | No hedging / uncertainty language |
| `citation_score` | Source URLs present in answer |
| `composite` | Average of all five |


In [ ]:
import pandas as pd
from evaluator import run_simple_evaluation, DEFAULT_TEST_SET

chain.reset()
print(f'Evaluating on {len(DEFAULT_TEST_SET)} questions:\n')

metrics = run_simple_evaluation(
    rag_chain   = chain,
    test_set    = DEFAULT_TEST_SET,
    output_path = 'data/eval_results.json',
    verbose     = True,
)

avgs = metrics['averages']
print('\n' + '=' * 50)
print('EVALUATION RESULTS')
print('=' * 50)
print(f"  Has answer rate : {avgs['avg_has_answer']:.1%}")
print(f"  Context score   : {avgs['avg_context_score']:.1%}")
print(f"  Relevance       : {avgs['avg_relevance']:.1%}")
print(f"  Faithfulness    : {avgs['avg_faithfulness']:.1%}")
print(f"  Citation score  : {avgs['avg_citation_score']:.1%}")
print(f"  Composite       : {avgs['avg_composite']:.1%}")
print('=' * 50)

df_eval = pd.DataFrame(metrics['per_question'])
df_eval['q'] = df_eval['question'].str[:55] + '...'
display(df_eval[['q', 'composite', 'relevance', 'faithfulness', 'citation_score']].round(3))


## Cell 17 — RAGAS Evaluation (Optional)

LLM-as-judge evaluation using RAGAS. Uncomment to run — uses ~10-20 API calls.

| Metric | What it measures |
|---|---|
| `faithfulness` | Answer entailed by retrieved context |
| `answer_relevancy` | Addresses the actual question |
| `context_precision` | Retrieved chunks are relevant |
| `context_recall` | Context covers the ground-truth answer |


In [ ]:
# Uncomment all lines below to run — uses Azure OpenAI credits (~10-20 calls)

# from evaluator import run_ragas_evaluation
# chain.reset()
# ragas_metrics = run_ragas_evaluation(
#     rag_chain   = chain,
#     test_set    = DEFAULT_TEST_SET,
#     output_path = 'data/ragas_eval_results.json',
# )
# print('\nRAGAS Results:')
# for k, v in ragas_metrics.items():
#     if isinstance(v, float):
#         print(f'  {k}: {v:.4f}')

print('RAGAS commented out. Uncomment above to run.')


## Cell 18 — Launch Streamlit Chatbot UI

Writes `ui/app.py` and launches it with a public ngrok URL. Open the printed link in a new tab. Leave this cell running — stopping it closes the UI.

In [ ]:
import os
os.makedirs('ui', exist_ok=True)

with open('ui/app.py', 'w') as f:
    f.write('...') # your existing app.py content — unchanged

print('✅ ui/app.py written')

In [ ]:
import subprocess, time, os
from pyngrok import ngrok, conf
from google.colab import userdata

# Kill any existing Streamlit on 8501
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
ngrok.kill()
time.sleep(2)

conf.get_default().auth_token = userdata.get('NGROK_TOKEN')

env = os.environ.copy()
env['PYTHONPATH'] = '/content/src'

proc = subprocess.Popen(
    ['streamlit', 'run', 'ui/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableXsrfProtection', 'false',
     '--server.enableWebsocketCompression', 'false',
     '--browser.serverAddress', '0.0.0.0'],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(8)

tunnel = ngrok.connect(
    addr=8501,
    proto="http",
    bind_tls=True,
)

url = tunnel.public_url
print(f'\n🌐 Chatbot live at: {url}')
print('Open the link above in a new tab.')
print('Leave this cell running — stopping it closes the UI.')

In [ ]:
import os
os.makedirs('ui', exist_ok=True)

app_code = r'''
import os
import sys
sys.path.insert(0, "/content/src")

os.environ.setdefault("CHROMA_PERSIST_DIR", "./data/chroma_db")
os.environ.setdefault("VECTOR_STORE_TYPE",  "chroma")
os.environ.setdefault("EMBEDDING_PROVIDER", "huggingface")
os.environ.setdefault("HF_EMBEDDING_MODEL", "BAAI/bge-small-en-v1.5")
os.environ.setdefault("RETRIEVAL_K",        "5")
os.environ.setdefault("LLM_TEMPERATURE",    "0.1")
os.environ.setdefault("LLM_MAX_TOKENS",     "1024")

import streamlit as st
from embedder    import load_vector_store
from llm_factory import get_llm
from rag_chain   import build_rag_chain

st.set_page_config(
    page_title="UChicago MS-ADS Assistant",
    page_icon="🎓",
    layout="wide",
    initial_sidebar_state="expanded",
)

st.markdown("""
<style>
  .uc-header {
    background: #800000;
    color: white;
    padding: 1.1rem 1.5rem;
    border-radius: 8px;
    margin-bottom: 1rem;
  }
  .uc-header h1 { margin: 0; font-size: 1.5rem; }
  .uc-header p  { margin: 0.2rem 0 0; opacity: 0.85; font-size: 0.9rem; }
  .source-tag {
    display: inline-block;
    background: #800000;
    color: white;
    font-size: 0.72rem;
    padding: 2px 8px;
    border-radius: 12px;
    margin: 2px 2px 0 0;
    text-decoration: none;
  }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="uc-header">
  <h1>🎓 MS in Applied Data Science — AI Assistant</h1>
  <p>University of Chicago | Ask me anything about the MS-ADS program</p>
</div>
""", unsafe_allow_html=True)

if "messages"     not in st.session_state: st.session_state.messages = []
if "rag_chain"    not in st.session_state: st.session_state.rag_chain = None
if "system_ready" not in st.session_state: st.session_state.system_ready = False
if "query_count"  not in st.session_state: st.session_state.query_count = 0

@st.cache_resource(show_spinner="Loading AI system — please wait...")
def load_system():
    vsm   = load_vector_store()
    llm   = get_llm()
    chain = build_rag_chain(vsm, llm)
    return chain

if not st.session_state.system_ready:
    st.session_state.rag_chain    = load_system()
    st.session_state.system_ready = True

with st.sidebar:
    st.markdown("### 🎓 MS-ADS Assistant")
    st.caption("University of Chicago")
    st.divider()

    st.markdown("**💡 Quick Questions**")
    quick_qs = [
        "What are the core courses?",
        "Admission requirements?",
        "Online vs in-person?",
        "Tell me about the capstone",
        "What are the tuition fees?",
        "Career outcomes for graduates?",
        "Application deadlines?",
        "Who are the instructors?",
    ]
    for q in quick_qs:
        if st.button(q, use_container_width=True, key=f"quick_{q}"):
            st.session_state["pending"] = q

    st.divider()
    col1, col2 = st.columns(2)
    with col1:
        if st.button("🗑️ Clear", use_container_width=True):
            st.session_state.messages    = []
            st.session_state.query_count = 0
            if st.session_state.rag_chain:
                st.session_state.rag_chain.reset()
            st.rerun()
    with col2:
        st.metric("Queries", st.session_state.query_count)

    st.divider()
    k_val = st.slider("Retrieved chunks (k)", 1, 10, 5)
    st.caption("More chunks = more context, slightly slower")

for msg in st.session_state.messages:
    avatar = "🧑" if msg["role"] == "user" else "🎓"
    with st.chat_message(msg["role"], avatar=avatar):
        st.markdown(msg["content"])
        if msg["role"] == "assistant" and msg.get("sources"):
            src_html = " ".join(
                f'<a class="source-tag" href="{s}" target="_blank">🔗 source</a>'
                for s in msg["sources"]
            )
            st.markdown(src_html, unsafe_allow_html=True)

if not st.session_state.messages:
    st.info(
        "**Welcome!** I can answer questions about the UChicago MS-ADS program — "
        "courses, admissions, tuition, instructors, capstone projects, and more. "
        "Use the Quick Questions in the sidebar or type your own below."
    )

pending    = st.session_state.pop("pending", None)
user_input = st.chat_input("Ask anything about the MS-ADS program...") or pending

if user_input:
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user", avatar="🧑"):
        st.markdown(user_input)

    with st.chat_message("assistant", avatar="🎓"):
        with st.spinner("Searching knowledge base..."):
            try:
                result  = st.session_state.rag_chain.ask(user_input, k=k_val)
                answer  = result["answer"]
                sources = result["sources"]
                docs    = result.get("docs", [])

                st.markdown(answer)

                if sources:
                    src_html = " ".join(
                        f'<a class="source-tag" href="{s}" target="_blank">🔗 source</a>'
                        for s in sources
                    )
                    st.markdown(src_html, unsafe_allow_html=True)

                if docs:
                    with st.expander(f"🔍 View {len(docs)} retrieved chunks"):
                        for i, doc in enumerate(docs, 1):
                            m = doc.metadata
                            st.markdown(
                                f"**Chunk {i}** | Section: `{m.get('section','')}` | "
                                f"[{m.get('page_title','')}]({m.get('source','')})"
                            )
                            st.text(doc.page_content[:400] + (" ..." if len(doc.page_content) > 400 else ""))
                            if i < len(docs):
                                st.divider()

            except Exception as exc:
                answer  = f"Sorry, an error occurred: {exc}"
                sources = []
                st.error(answer)

    st.session_state.messages.append({
        "role":    "assistant",
        "content": answer,
        "sources": sources,
    })
    st.session_state.query_count += 1
    st.rerun()
'''

with open('ui/app.py', 'w') as f:
    f.write(app_code.strip())

print('✅ ui/app.py written successfully')

# Verify it looks right
with open('ui/app.py') as f:
    lines = f.readlines()
print(f'   {len(lines)} lines written')
print(f'   First line: {lines[0].strip()}')

In [ ]:
# (unused)


In [ ]:
# (unused)


In [ ]:
# (unused)


In [ ]:
# (unused)
